# L3_Clustering_MonteCarlo.ipynb 修改说明

本文档说明为了跑通 `L3_Clustering_MonteCarlo.ipynb` 所做的修改。目标 notebook 使用 Stata 内核，主要内容是通过 Monte Carlo 模拟比较 iid、robust 和 cluster 标准误在面板相关误差下的表现。

## 1. 运行环境问题

- 已在当前目录创建局部 Python 环境 `.venv`，并安装 `nbformat`、`nbclient`、`jupyter_client`、`ipykernel`、`stata_kernel`。
- 本机 Stata 路径为 `D:\Stata16SE Win64\Stata\StataSE-64.exe`。
- 已使用 Stata 16 SE 批处理方式执行从 notebook 提取出的 Stata 代码。

## 2. 发现的主要问题

### 问题 A：`xtset` 之前使用了 `L.eps`

原代码在 `xtset unit t` 之前执行：

```stata
replace eps = `rho' * L.eps + rnormal(0, 1) if t > 1 & !missing(L.eps)
```

Stata 的 `L.` 滞后算子依赖已声明的时间序列或面板设定。没有先 `xtset` 时，这一步会报类似 `time variable not set` 的错误。

### 问题 B：时间效应生成逻辑会产生大量缺失值

原代码只在 `t == 1` 时生成 `time_fe`，然后按 `t` 求均值：

```stata
gen time_fe = rnormal(0, 0.5) if t == 1
bysort t: egen tfe = mean(time_fe)
```

这样只有第 1 期有非缺失时间效应，其他期的 `time_fe` 都是缺失值，最终 `Y` 在大多数时期也会缺失，影响后续回归和 Monte Carlo 结果。

### 问题 C：重复运行时 `program define gen_panel` 会报错

如果 notebook 单元被反复执行，Stata 中已经存在同名 program，直接再次 `program define gen_panel` 会报 program already defined。

### 问题 D：`boottest` 的 `boottype(wild-t)` 选项不稳

`boottest` 的 `boottype()` 常见取值是 `wild` 或 `score`，而不是 `wild-t`。这里改为直接使用默认 wild bootstrap，并保留聚类和重复次数设置。

另外，Stata 16 中的 `boottest` 不能直接接在吸收多组固定效应的 `reghdfe` 后面运行。小聚类示例只有 300 行，因此改用显式虚拟变量回归后再运行 `boottest`。

## 3. 已做修改

### 修改 1：包安装顺序

将 `ftools` 放在 `reghdfe` 前安装，因为 `reghdfe` 依赖 `ftools`。

### 修改 2：允许重复定义 `gen_panel`

在 `program define gen_panel` 前增加：

```stata
capture program drop gen_panel
```

这样 notebook 可以重复运行，不会因为同名 program 已存在而中断。

### 修改 3：修复时间效应生成

改为每个时期生成一个随机时间效应，并复制给同一期的所有单位：

```stata
sort t unit
by t: gen time_fe = rnormal(0, 0.5) if _n == 1
by t: replace time_fe = time_fe[1]
```

### 修改 4：修复 AR(1) 误差生成

改为先声明面板，再在每个单位内部递推误差：

```stata
xtset unit t
gen eps = .
by unit (t): replace eps = rnormal(0, 1) if _n == 1
by unit (t): replace eps = `rho' * eps[_n-1] + rnormal(0, 1) if _n > 1
```

这样避免在 `xtset` 前使用 `L.`，也避免重复生成同一个误差项。

### 修改 5：增强 rho 分组结果输出兼容性

将较依赖 Stata 版本的：

```stata
table rho, stat(mean reject_iid reject_cl)
```

改为：

```stata
collapse (mean) reject_iid reject_cl, by(rho)
format reject_iid reject_cl %6.3f
list rho reject_iid reject_cl, noobs
```

这样在更多 Stata 版本中都可以输出各 rho 下的平均拒绝率。

### 修改 6：修复 Wild Cluster Bootstrap 调用

将：

```stata
boottest D, boottype(wild-t) reps(999) cluster(unit)
```

改为：

```stata
regress Y D i.unit i.t, vce(cluster unit)
boottest D, reps(999) cluster(unit) seed(20260516)
```

保留 wild cluster bootstrap 的核心设定，并加上随机种子保证结果可复现。这里使用显式的 `i.unit i.t`，是因为 `boottest` 在 Stata 16 中不能直接跟在吸收两组固定效应的 `reghdfe` 后面。

## 4. 建议的最终验证方式

已使用 Stata 16 SE 批处理运行 notebook 的 Stata 代码单元。完整运行产生了 `mc_cluster.dta` 与 `mc_rho.dta`，并生成日志文件 `run_L3_Clustering_MonteCarlo.log`。

主 Monte Carlo 单元输出的拒绝率为：iid 28.7%，HC1 29.3%，cluster 6.3%。rho 分组结果为：rho=0 时 iid 4.0%、cluster 3.0%；rho=0.4 时 iid 14.0%、cluster 3.0%；rho=0.8 时 iid 20.0%、cluster 5.0%。